## GPT-2 small

In [1]:
from gpt_config import GPTConfig

gpt_124M_config = GPTConfig(
    vocab_size=50257,
    context_length=1024,
    embedding_dim=768,
    n_heads=12,
    n_layers=12,
    dropout=0.1,
    qkv_bias=False
)

In [2]:
import torch
import tiktoken

from gpt_model import GPTModel


tokenizer = tiktoken.get_encoding('gpt2')
txt1 = 'Every day moves you'
txt2 = 'Every day holds a'

batch = torch.stack([
    torch.tensor(tokenizer.encode(txt1)),
    torch.tensor(tokenizer.encode(txt2))
])
print(batch)
print()

model = GPTModel(gpt_124M_config)
logits = model(batch)
print(logits.shape) # 4 tokens in each batch
print(logits)


tensor([[6109, 1110, 6100,  345],
        [6109, 1110, 6622,  257]])

torch.Size([2, 4, 50257])
tensor([[[-1.1586, -1.0219,  0.3143,  ..., -0.1154,  0.4042,  0.4390],
         [-0.6717,  0.2385, -1.0611,  ...,  0.1498, -0.4995,  0.6643],
         [-0.9578, -0.1389, -0.2419,  ...,  0.1505,  0.6583, -0.6926],
         [-0.1753, -0.8495, -0.1605,  ...,  0.3860, -0.1907,  0.0831]],

        [[-0.4459, -0.2556, -0.0932,  ..., -0.4502,  0.8388,  0.7359],
         [-0.2584,  0.4667, -0.8907,  ...,  0.4199, -0.7047,  0.2285],
         [-0.5009, -0.5251, -0.4515,  ..., -0.1026,  0.5136, -0.5386],
         [-0.1856, -0.4945, -0.7880,  ...,  0.0456, -0.3865, -0.0761]]],
       grad_fn=<UnsafeViewBackward0>)


## Layer normalization
Adresses the problem of vanishing and exploding gradients.\
Typically, it's applied befor and after the multi-head attention module.

BatchNorm makes feature distributions consistent across examples. LayerNorm makes the scale of each token's representation consistent, independently of the batch.

In [3]:
from layer_norm import LayerNorm

ln = LayerNorm(embedding_dim=5)
batch_example = torch.randn(2, 5)
output = ln(batch_example)
mean = output.mean(dim=-1, keepdim=True)
var = output.var(dim=-1, keepdim=True, unbiased=False)
print(f'Mean: {mean}')
print(f'Variance: {var}')

Mean: tensor([[-2.3842e-08],
        [-4.3958e-08]], grad_fn=<MeanBackward1>)
Variance: tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


## GELU Activation
Gaussian error linear unit

$GELU(x) = x \cdot \Phi(x)$, 

where $\Phi(x)$ is the cumulative distribution function of the standard Gaussian distribution.

It is a smooth, nonlinear function that approximatesReLU, but with a non-zero gradient for almost all negative values (except ~-0.75). 

Neurons that receive negative input can still contribute to the learning process.

## Shortcut Connections

A.k.a. skip residual connections.

Vanishing gradients - the issue where gradients become progressively smaller as they propagate backwards through the layers, making it difficult to effectively train earlier layers. 

Alternative, shorter path that skips one or more, is achieved by adding the output of one layer to the output of a later layer. 

Encoder–decoder skip connections (U-Net) - different kind of skipped connections with concatenation instead of addition. The main purpose is to preserve spatial information. 

## Transformer Block
The self attention mechanism in the multi-head attention block identifies and analyses relationships between elements in the input sequence. \ 
The feed forward network modifies the data individually at each position. 

In [4]:
from transformer_block import TransformerBlock

x = torch.randn(2, 10, 768)
transformer_block = TransformerBlock(gpt_124M_config)
output = transformer_block(x)
print(x.shape)
print(output.shape)

torch.Size([2, 10, 768])
torch.Size([2, 10, 768])
